# Stage 02a — Preprocessing (review-batch validation & cleanup)

Runs **between** the human review export (`reviewed_patents_<batch>.xlsx`,
produced by the HTML wizard off `01a_wizard_feed`) and `02b_postprocessing`.

Pipeline: **load** the raw wizard export (resolving blank `Image_Path`s
in-memory via `scripts/resolve_image_paths.py` logic — the raw file is never
touched) → **validate** (Rule A Combined Thrust, Rule B Fixed Empennage,
Rule C duplicate-chain `UAVSimilar` propagation) → **human pass** over the
flagged queue in the ipywidgets UI → **export** a brand-new timestamped
`preprocessed/` xlsx (formatted per `Rearranging th eexcell.py`) for
`02b_postprocessing`.

Format contract (differs from `excel_schema.py`'s source format!):
7 columns, `Value`s are `"ID — Label"` composites, M3 kinematics are
card-prefixed (`wing1_propKin`, ...), edge tags live in `META/t1EdgeTags`.


## Section 1 — Imports & Config

In [ ]:
import sys
from pathlib import Path
from datetime import datetime

repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd
import networkx as nx          # Rule C — duplicate-chain connected components
import openpyxl
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
import ipywidgets as widgets
from IPython.display import display, clear_output

from src.config_loader import load_config
import src.processor as proc   # Section 5 — parse_arch_id() for image lookup

cfg = load_config()

sheet_name = "Batch_01"   # <- the batch to preprocess


## Section 2 — "Rearranging the excell" layout logic (reusable)

Ports `src/Rearranging th eexcell.py`'s three visual behaviors into one
function we call from Section 6:

1. **Cell merging** — for every column, consecutive rows sharing the same
   value are merged into one cell (vertical-center, wrap-text), exactly as
   the original script's "merge consecutive rows with identical values" step.
2. **Column widths** — the original hardcoded a `{'A': 18, 'B': 10, ...}`
   letter->width dict tied to one fixed 7-column layout (`Patent_ID, Section,
   Sub_Dimension, Field, Value, Source, Image_Path`). Our schema carries more
   columns (`Definition`, `Options`, `Confidence`, `Needs_Review`,
   `pre_process_flags`) and the column order can shift, so a hardcoded letter
   map would silently mis-size or ignore columns. Instead we **auto-fit**:
   each column's width is `max(min_width, min(longest_value + 2, max_width))`,
   computed from that column's actual content.
3. **Text wrap / alignment** — final pass sets `wrap_text=True` on every
   data cell, `vertical='center'` for merged cells and `'top'` otherwise,
   matching the original.

The original script also **truncates** long `Value` cells (`abstract` /
`description_of_drawings`) and `Image_Path` cells for on-screen compactness.
That's a real data loss if baked into the file handed to `02b_postprocessing`,
so it's kept but off by default (`truncate_long_text=False`).


In [ ]:
def format_review_workbook(
    df: pd.DataFrame,
    output_xlsx: Path,
    truncate_long_text: bool = False,
    min_col_width: int = 8,
    max_col_width: int = 60,
) -> None:
    """Write df to output_xlsx as TWO sheets:

    - "Review"  — flat, machine-readable, NO cell merging: what
                  02b_postprocessing / proc.load_review_images consume.
                  (Merged cells read back as NaN in pandas — merging the data
                  sheet silently destroys Patent_ID/Value on round-trip.)
    - "Compact" — the human view per src/Rearranging th eexcell.py:
                  consecutive identical cells merged, wrap text, auto-fit.

    Styling uses shared Alignment/Font/Fill objects — constructing one per
    cell makes openpyxl take minutes on a 20k-row batch.
    """
    headers = list(df.columns)
    truncate_fields = {"abstract", "description_of_drawings"}
    header_fill = PatternFill(start_color="F2F2F2", end_color="F2F2F2", fill_type="solid")
    header_font = Font(bold=True)
    align_top = Alignment(vertical="top", wrap_text=True)
    align_center = Alignment(vertical="center", wrap_text=True)

    def _cell_value(row, col_name):
        val = row[col_name]
        if pd.isna(val):
            return None
        val_str = str(val)
        if truncate_long_text:
            if col_name == "Value" and str(row.get("Field")) in truncate_fields and len(val_str) > 25:
                val_str = val_str[:25] + "..."
            if col_name == "Image_Path" and len(val_str) > 15:
                val_str = val_str[:15] + "..."
        return val_str

    def _fill_sheet(ws, merge: bool):
        ws.append(headers)
        for col_num in range(1, len(headers) + 1):
            cell = ws.cell(row=1, column=col_num)
            cell.fill = header_fill
            cell.font = header_font

        for _, row in df.iterrows():
            ws.append([_cell_value(row, c) for c in headers])

        # Wrap-text/top-align every data cell up front (shared object, cheap);
        # merge anchors get re-set to center below.
        for row_cells in ws.iter_rows(min_row=2):
            for cell in row_cells:
                cell.alignment = align_top

        if merge:
            # Merge consecutive rows with identical values, per column.
            for col in range(1, ws.max_column + 1):
                start_row = 2
                for row in range(3, ws.max_row + 1):
                    val_prev = ws.cell(row=row - 1, column=col).value
                    val_curr = ws.cell(row=row, column=col).value
                    if val_curr != val_prev or val_prev is None:
                        if (row - 1) > start_row and val_prev is not None:
                            ws.merge_cells(start_row=start_row, start_column=col, end_row=row - 1, end_column=col)
                            ws.cell(row=start_row, column=col).alignment = align_center
                        start_row = row
                if ws.max_row > start_row and ws.cell(row=start_row, column=col).value is not None:
                    ws.merge_cells(start_row=start_row, start_column=col, end_row=ws.max_row, end_column=col)
                    ws.cell(row=start_row, column=col).alignment = align_center

        # Auto-fit column widths from actual content (the original script's
        # hardcoded {'A': 18, ...} assumed one fixed 7-column layout).
        for col_num, col_name in enumerate(headers, start=1):
            col_letter = get_column_letter(col_num)
            longest = max(
                [len(col_name)] + [len(str(v)) for v in df[col_name].dropna().astype(str)],
                default=len(col_name),
            )
            ws.column_dimensions[col_letter].width = max(min_col_width, min(longest + 2, max_col_width))

    wb = openpyxl.Workbook()
    ws_data = wb.active
    ws_data.title = "Review"
    _fill_sheet(ws_data, merge=False)
    _fill_sheet(wb.create_sheet("Compact"), merge=True)

    wb.save(output_xlsx)
    print(f"Successfully generated formatted workbook (Review + Compact): {output_xlsx}")


## Section 3 — Data Loading

Read the raw `reviewed_patents_<batch>.xlsx` export from the review stage.
Never mutate/overwrite the original export file — all cleaning happens on an
in-memory copy, written out as a new file in Section 6.


In [ ]:
REVIEWED_XLSX = Path(cfg["paths"]["html_review_exports"]) / f"reviewed_patents_{sheet_name}.xlsx"

assert REVIEWED_XLSX.exists(), (
    f"{REVIEWED_XLSX} not found. Export it from the wizard (\"Export batch\" "
    f"button) first."
)

# Long format: one row per (Patent_ID, Section, Sub_Dimension, Field, Value,
# Source, Image_Path). NOTE the wizard export is NOT the excel_schema.py
# source format: Values are "ID — Label" composites (use _strip_label), M3
# fields are card-prefixed (wing1_propKin, boom_t1_propKin, ...), and the
# edge-case tags live in META/t1EdgeTags.
df_raw = pd.read_excel(REVIEWED_XLSX, sheet_name="Review")
df = df_raw.copy()   # all downstream work happens on this copy

print(f"loaded {len(df)} rows from {REVIEWED_XLSX.name}")

# ── Resolve blank Image_Path cells in-memory (scripts/resolve_image_paths.py) ──
# The wizard's "📎 Attach/Replace Image" can only record a filename, never a
# disk path, so those rows come back with a blank Image_Path. The script
# fixes the file in place; here we apply the same find_image() lookup to the
# in-memory copy instead, so the raw export stays untouched and Section 6's
# output carries resolved paths.
#
# "Image: (fig N)" rows are text-referenced figures with no matched image on
# disk — placeholders by design, skipped so the unresolved report only shows
# genuinely missing files (e.g. clipboard pastes never saved into matched/).
from scripts.resolve_image_paths import find_image

matched_root = Path(cfg["paths"]["matched"])
img_names = df["Sub_Dimension"].astype(str).str.removeprefix("Image: ").str.strip()
is_placeholder = img_names.str.match(r"^\(fig .*\)$") | (img_names == "(none available)")
img_mask = (
    (df["Section"] == "T2")
    & df["Sub_Dimension"].astype(str).str.startswith("Image: ")
    & ~is_placeholder
    & (df["Image_Path"].isna() | (df["Image_Path"].astype(str).str.strip() == ""))
)
resolved, unresolved = 0, []
for idx, row in df[img_mask].iterrows():
    fname = img_names[idx]
    hit = find_image(matched_root, str(row["Patent_ID"]).strip(), fname)
    if hit:
        df.at[idx, "Image_Path"] = str(hit)
        resolved += 1
    else:
        unresolved.append((row["Patent_ID"], fname))

print(f"Image_Path resolution: {int(img_mask.sum())} blank (placeholders excluded) | "
      f"{resolved} resolved | {len(unresolved)} unresolved")
if unresolved:
    print("  ⚠ unresolved — copy these files into the patent's matched/ folder and re-run:")
    for pid, fname in sorted(set(unresolved)):
        print(f"    - {pid}: {fname}")


## Section 4 — Validation Rule Logic

In [ ]:
def _strip_label(value):
    """Wizard export Values are "ID — Label" composites (withLabel format,
    e.g. "TP — Vectored Thrust — Tilt Propulsors"). Return just the ID part;
    None for NaN/blank. Mirrors the HTML's stripLabel().
    """
    if pd.isna(value):
        return None
    return str(value).split(" — ")[0].strip()


def _append_flag(df: pd.DataFrame, patent_ids, message: str) -> None:
    """In-place: append `message` to pre_process_flags for every row whose
    Patent_ID is in patent_ids (idempotent — skips patents that already
    carry that exact message).
    """
    if "pre_process_flags" not in df.columns:
        df["pre_process_flags"] = ""
    df["pre_process_flags"] = df["pre_process_flags"].fillna("")

    mask = df["Patent_ID"].isin(patent_ids)
    df.loc[mask, "pre_process_flags"] = df.loc[mask, "pre_process_flags"].apply(
        lambda existing: existing if message in existing else f"{existing}{message}"
    )


### Rule A — Combined Thrust (flag for review, target = CVT)

"Combined Thrust" **is** the existing G1 option `CVT — Vectored Thrust —
Combined` — no new taxonomy id needed.

For each `Patent_ID`, compare its stripped `topType` (Section `G1`) against
its per-card kinematics rows (`Field` ends with `_propKin`: `wing1_propKin`,
`fuselage_propKin`, `boom_t1_propKin`, ...; stripped ids
`Fixed | Tilt | Vectored | Cyclic`):

- `topType == TP` **and** propKin mixes `Fixed` + tilting → the patent looks
  like combined thrust mislabelled as pure tilt-propulsor. **Flag only**
  (`"Review: Potential Combined Thrust (TP→CVT?);"`) — the Section 5 UI shows
  the image and the reviewer decides via **Update to CVT** / **Keep As Is**.
- `topType == CVT` with mixed propKin is *consistent* — no flag.

The rule deliberately does not auto-overwrite: since the original labels may
simply be wrong, the human pass in Section 5 is the decision point.


In [ ]:
CVT_VALUE = "CVT — Vectored Thrust — Combined"   # the wizard's own composite for CVT


def _flag_combined_thrust(df: pd.DataFrame) -> pd.DataFrame:
    """Rule A — Combined Thrust candidates (flag-only; human decides in the
    Section 5 UI whether to switch topType TP → CVT).

    Flags patents whose stripped topType is TP while their *_propKin rows mix
    "Fixed" with a tilting mechanism (Tilt/Vectored/Cyclic). CVT patents with
    mixed propKin are already consistent and are left alone.
    """
    df = df.copy()

    top_type_by_patent = (
        df.loc[df["Field"] == "topType"]
        .set_index("Patent_ID")["Value"].map(_strip_label)
    )

    propkin_rows = df.loc[df["Field"].astype(str).str.endswith("_propKin")]

    def _is_mixed(values: pd.Series) -> bool:
        vals = {_strip_label(v) for v in values} - {None}
        return "Fixed" in vals and bool(vals - {"Fixed"})

    mixed_by_patent = propkin_rows.groupby("Patent_ID")["Value"].apply(_is_mixed)

    flagged_ids = [
        pid for pid, is_mixed in mixed_by_patent.items()
        if is_mixed and top_type_by_patent.get(pid) == "TP"
    ]

    if flagged_ids:
        _append_flag(df, flagged_ids, "Review: Potential Combined Thrust (TP→CVT?);")

    return df


### Rule B — Fixed Empennage

For each `Patent_ID`, if every `propKin` row's Value is `"Fixed"` (i.e. no
tilting/vectored/cyclic mechanism anywhere — pure multirotors/screws,
`topType` `RC`/`MR`, or any winged layout that happens to have none of its
propulsors tilt), force that patent's `empKin` row (`Field == "empKin"`,
Section `M2`) Value to `"Fixed"` and append
`"Empennage forced to Fixed;"` to `pre_process_flags`.

Patents with no `empKin` row at all (e.g. `Tailless` empType, so no tail to
lock) are left alone — there's nothing to force.


In [ ]:
def _flag_fixed_empennage(df: pd.DataFrame) -> pd.DataFrame:
    """Rule B — Fixed Empennage.

    Patents with no tilting propulsor (all *_propKin rows strip to "Fixed")
    get their empKin Value forced to "Fixed" (only if it isn't already), and
    are flagged. Patents without any empKin row (Tailless) are untouched.
    """
    df = df.copy()

    propkin_rows = df.loc[df["Field"].astype(str).str.endswith("_propKin")]
    has_tilt_by_patent = propkin_rows.groupby("Patent_ID")["Value"].apply(
        lambda values: any(_strip_label(v) not in (None, "Fixed") for v in values)
    )
    zero_tilt_ids = has_tilt_by_patent[~has_tilt_by_patent].index

    emp_kin_mask = (
        (df["Field"] == "empKin")
        & df["Patent_ID"].isin(zero_tilt_ids)
        & (df["Value"].map(_strip_label) != "Fixed")
    )
    forced_ids = df.loc[emp_kin_mask, "Patent_ID"].unique().tolist()

    if forced_ids:
        df.loc[emp_kin_mask, "Value"] = "Fixed"
        _append_flag(df, forced_ids, "Empennage forced to Fixed;")

    return df


### Rule C — Duplicate Chain Tagging

The duplicate link lives in the T1 fields `isDuplicate` (bool) +
`duplicateId` (Value = the plain Patent_ID it duplicates — no label suffix).
Chains (A dup-of B dup-of C) are resolved as connected components of an
undirected graph built from those `duplicateId` edges via `networkx`.

The "UAV, but similar enough" tag (`UAVSimilar`) is an **edge-case tag**, not
a disapproval reason: it lives in the `META` section, Field `t1EdgeTags`,
Sub_Dimension `"Review Metadata"`. If any patent in a connected component
carries `UAVSimilar` there, every patent in that component gets it too
(appended comma-separated to an existing t1EdgeTags row, or a new row is
created) and is flagged with `"UAV Tag Propagated;"`.


In [ ]:
def _propagate_duplicate_tag(df: pd.DataFrame) -> pd.DataFrame:
    """Rule C — Duplicate Chain Tagging.

    Builds an undirected graph of Patent_ID <-> duplicateId edges, finds
    connected components (chains), and — if any patent in a component carries
    "UAVSimilar" in its META/t1EdgeTags row — stamps that tag onto every
    patent in the component (adding the row if it doesn't already exist).
    """
    df = df.copy()

    graph = nx.Graph()
    graph.add_nodes_from(df["Patent_ID"].astype(str).unique())

    dup_edges = df.loc[
        (df["Field"] == "duplicateId") & df["Value"].notna() & (df["Value"].astype(str).str.strip() != "")
    ]
    for _, row in dup_edges.iterrows():
        graph.add_edge(str(row["Patent_ID"]), str(row["Value"]).strip())

    # "UAV, but similar enough" lives in the META section's t1EdgeTags field
    # (Value contains the tag id "UAVSimilar") — NOT in t1DisapproveReason.
    edge_tag_rows = df["Field"] == "t1EdgeTags"
    tagged_ids = set(
        df.loc[edge_tag_rows & df["Value"].astype(str).str.contains("UAVSimilar", na=False), "Patent_ID"].astype(str)
    )

    # Any chain (component of size > 1) that contains a tagged patent needs
    # the tag propagated to its other members.
    propagate_ids = set()
    for component in nx.connected_components(graph):
        if len(component) > 1 and (component & tagged_ids):
            propagate_ids |= (component - tagged_ids)

    if propagate_ids:
        in_chain = df["Patent_ID"].astype(str).isin(propagate_ids)

        # Existing t1EdgeTags rows: append the tag (comma-separated) unless present.
        existing_mask = edge_tag_rows & in_chain
        df.loc[existing_mask, "Value"] = df.loc[existing_mask, "Value"].apply(
            lambda v: "UAVSimilar" if pd.isna(v) or not str(v).strip()
            else (str(v) if "UAVSimilar" in str(v) else f"{v},UAVSimilar")
        )

        # Patents in the chain with no t1EdgeTags row yet — add one, matching
        # the wizard export's 7-column schema.
        already_updated = set(df.loc[existing_mask, "Patent_ID"].astype(str))
        missing_ids = propagate_ids - already_updated
        if missing_ids:
            new_rows = pd.DataFrame([
                {
                    "Patent_ID": pid, "Section": "META", "Sub_Dimension": "Review Metadata",
                    "Field": "t1EdgeTags", "Value": "UAVSimilar",
                    "Source": "rule_c_propagation", "Image_Path": None, "pre_process_flags": "",
                }
                for pid in missing_ids
            ])
            df = pd.concat([df, new_rows], ignore_index=True)

        _append_flag(df, propagate_ids, "UAV Tag Propagated;")

    return df


### Rule D — Duplicate Inheritance (topType + isApproved from chain root)

The HTML wizard auto-copies M1–M3 to duplicates but **not G1**, and doesn't
force an approval choice on them. Verified on Batch_01: ~131 approved patents
missing `topType` and all 32 NaN-`isApproved` patents are duplicates whose
chain root carries the value.

For every member of a duplicate chain (per Rule C's graph, following
`duplicateId` links to the chain root):

- missing/blank `topType` → copy the root's `topType` (new G1 row,
  `Source = "rule_d_inherited"`), flag `"G1 inherited from duplicate root;"`.
- `isApproved` NaN → copy the root's `isApproved`, flag
  `"Approval inherited from duplicate root;"`.

Members whose root also lacks the value, and non-duplicates missing topType,
are flagged `"Missing topType — needs manual review;"` for the Section 5 UI
instead (nothing to inherit).

The 41 `_archN`-suffixed ids lacking `isApproved` rows are left alone —
approval lives on the base patent by design.


In [ ]:
def _resolve_chain_root(patent_id: str, dup_target: pd.Series) -> str:
    """Follow duplicateId links to the chain root (cycle-safe)."""
    seen = set()
    current = str(patent_id)
    while current not in seen:
        seen.add(current)
        target = dup_target.get(current)
        if target is None or pd.isna(target) or not str(target).strip():
            return current
        current = str(target).strip()
    return current  # cycle — return where we stopped


def _inherit_from_duplicate_root(df: pd.DataFrame) -> pd.DataFrame:
    """Rule D — duplicates inherit topType (G1) and isApproved from their
    chain root when they lack a value of their own.

    "Missing topType" is only flagged where it's actually unexpected:
    disapproved patents skip G1 by design, and multi-arch patents carry
    topType on their _archN ids, not the base id.
    """
    df = df.copy()

    dup_rows = df.loc[(df["Field"] == "duplicateId") & df["Value"].notna()
                      & (df["Value"].astype(str).str.strip() != "")]
    dup_target = dup_rows.set_index(dup_rows["Patent_ID"].astype(str))["Value"]
    is_dup_ids = set(dup_target.index)

    top_type_rows = df.loc[df["Field"] == "topType"]
    top_type_by_patent = top_type_rows.set_index(top_type_rows["Patent_ID"].astype(str))["Value"]
    appr_rows = df.loc[df["Field"] == "isApproved"]
    appr_by_patent = appr_rows.set_index(appr_rows["Patent_ID"].astype(str))["Value"]

    all_id_strings = df["Patent_ID"].astype(str).unique()
    # arch-suffixed ids never carry their own isApproved (approval lives on
    # the base patent), and multi-arch BASE ids never carry their own topType
    # (it lives on the _archN ids) — both excluded from "missing" checks.
    arch_bases = {p.split("_arch")[0] for p in all_id_strings if "_arch" in p}
    all_ids = {p for p in all_id_strings if "_arch" not in p}

    def _is_disapproved(pid):
        return str(appr_by_patent.get(pid)).strip().lower() == "false"

    new_rows, tt_inherited, appr_inherited, needs_manual = [], [], [], []
    for pid in sorted(all_ids):
        needs_tt = (pd.isna(top_type_by_patent.get(pid))
                    and pid not in arch_bases and not _is_disapproved(pid))
        has_appr = pd.notna(appr_by_patent.get(pid))
        if not needs_tt and has_appr:
            continue

        if pid in is_dup_ids:
            root = _resolve_chain_root(pid, dup_target)
            root_tt = top_type_by_patent.get(root)
            root_appr = appr_by_patent.get(root)

            if needs_tt and pd.notna(root_tt):
                new_rows.append({
                    "Patent_ID": pid, "Section": "G1", "Sub_Dimension": "Topology Type",
                    "Field": "topType", "Value": root_tt,
                    "Source": "rule_d_inherited", "Image_Path": None, "pre_process_flags": "",
                })
                tt_inherited.append(pid)
            elif needs_tt:
                needs_manual.append(pid)

            if not has_appr and pd.notna(root_appr):
                appr_mask = (df["Field"] == "isApproved") & (df["Patent_ID"].astype(str) == pid)
                if appr_mask.any():
                    df.loc[appr_mask, "Value"] = root_appr
                else:
                    new_rows.append({
                        "Patent_ID": pid, "Section": "T1", "Sub_Dimension": "T1 — Approval Status",
                        "Field": "isApproved", "Value": root_appr,
                        "Source": "rule_d_inherited", "Image_Path": None, "pre_process_flags": "",
                    })
                appr_inherited.append(pid)
        elif needs_tt:
            # Non-duplicate, approved, single-arch, no topType — needs eyes.
            needs_manual.append(pid)

    if new_rows:
        df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
    if tt_inherited:
        _append_flag(df, tt_inherited, "G1 inherited from duplicate root;")
    if appr_inherited:
        _append_flag(df, appr_inherited, "Approval inherited from duplicate root;")
    if needs_manual:
        _append_flag(df, needs_manual, "Missing topType — needs manual review;")

    print(f"Rule D: topType inherited {len(tt_inherited)} | isApproved inherited "
          f"{len(appr_inherited)} | needs manual review {len(needs_manual)}")
    return df


### Orchestration

`run_validation_rules` runs, in order:

0. **Ghost-block cleanup** — when the same image (`Patent_ID` +
   `Sub_Dimension`) carries the same Field twice with different values (a
   stale stub left from re-keying a figure in the wizard, e.g.
   US2018339773A1's `figKey 9/disapproved` vs `1F/approved`), keep the
   **last** occurrence (the later, complete block) and drop the stub rows.
1. Rule A — flag TP patents with mixed propKin as CVT candidates.
2. Rule B — force empKin to Fixed on zero-tilt patents.
3. Rule C — propagate `UAVSimilar` across duplicate chains.
4. Rule D — duplicates inherit topType/isApproved from their chain root.

Returns a new DataFrame with a `pre_process_flags` column for the Section 5
review UI; the input df is never mutated.


In [ ]:
def _drop_ghost_image_blocks(df: pd.DataFrame) -> pd.DataFrame:
    """Step 0 — drop stale duplicate rows within an image block.

    The wizard can leave a stub behind when a figure is re-keyed (same
    Patent_ID + "Image: <file>" Sub_Dimension, same Field appearing twice
    with different values). Blocks are written in save order, so the LAST
    occurrence is the current one — keep it, drop earlier ones.
    """
    df = df.copy()
    is_img = df["Sub_Dimension"].astype(str).str.startswith("Image: ")
    key_cols = ["Patent_ID", "Sub_Dimension", "Field"]
    dup_mask = is_img & df.duplicated(subset=key_cols, keep="last")
    if dup_mask.any():
        ghosts = df.loc[dup_mask, key_cols]
        ghost_patents = ghosts["Patent_ID"].unique().tolist()
        print(f"ghost-block cleanup: dropping {int(dup_mask.sum())} stale row(s) "
              f"for {ghost_patents}")
        df = df.loc[~dup_mask].reset_index(drop=True)
        _append_flag(df, ghost_patents, "Stale image block removed;")
    return df


def run_validation_rules(df: pd.DataFrame) -> pd.DataFrame:
    """Cleanup + Rules A–D on a long-format wizard-export df.

    Returns a new DataFrame — the input df is never mutated. Adds/updates a
    'pre_process_flags' column (semicolon-terminated messages, one per rule
    that touched a given Patent_ID) for the Section 5 review UI to surface.
    """
    df = _drop_ghost_image_blocks(df)
    df = _flag_combined_thrust(df)
    df = _flag_fixed_empennage(df)
    df = _propagate_duplicate_tag(df)
    df = _inherit_from_duplicate_root(df)
    return df


# df = run_validation_rules(df)
# df["pre_process_flags"].value_counts()


## Section 5 — Interactive Review UI

One flagged patent at a time. Left pane renders its main architecture image
from `matched/<batch>/<patent_id>_*/`; right pane shows Patent ID, Company,
Architecture (both from `data/batches.xlsx`, keyed on the base patent id —
see `src/grouper.py`), and the `pre_process_flags` from Section 4.

- **Update to Combined Thrust** / **Keep As Is** both record the reviewer's
  decision into `pre_process_flags` and advance to the next flagged record;
  the former also (re)writes `topType` to `"Combined Thrust"`.
- **Prev** / **Next** move through the queue without changing anything.
- The search box filters the queue by substring match against Company,
  Architecture, or Patent ID (case-insensitive).
- A missing/unreadable image never raises — the image pane shows a
  short warning instead.


In [ ]:
matched_dir = Path(cfg["paths"]["matched"]) / sheet_name


def build_issue_queue(df: pd.DataFrame, cfg: dict, sheet_name: str) -> pd.DataFrame:
    """One row per flagged Patent_ID: flags text, current topType, and
    Company/Architecture pulled from data/batches.xlsx (keyed on the base,
    non-arch-suffixed patent id — see src/grouper.py).
    """
    flags_col = df.get("pre_process_flags")
    if flags_col is None:
        return pd.DataFrame(columns=["Patent_ID", "flags", "topType", "Company", "Architecture"])

    flagged_ids = df.loc[flags_col.astype(str).str.len() > 0, "Patent_ID"].unique()
    queue = pd.DataFrame({"Patent_ID": flagged_ids})

    flags_by_patent = (
        df[df["Patent_ID"].isin(flagged_ids)]
        .groupby("Patent_ID")["pre_process_flags"]
        .apply(lambda s: max(s.dropna(), key=len, default=""))
    )
    top_type_by_patent = df.loc[df["Field"] == "topType"].set_index("Patent_ID")["Value"]
    queue["flags"] = queue["Patent_ID"].map(flags_by_patent)
    queue["topType"] = queue["Patent_ID"].map(top_type_by_patent)

    base_ids = queue["Patent_ID"].map(lambda pid: proc.parse_arch_id(pid)[0])
    try:
        batches_df = pd.read_excel(Path(cfg["paths"]["data"]) / "batches.xlsx", sheet_name=sheet_name, dtype=str)
        meta = batches_df.drop_duplicates("patent_id").set_index("patent_id")
        queue["Company"] = base_ids.map(meta.get("company_canonical", pd.Series(dtype=str)))
        queue["Architecture"] = base_ids.map(meta.get("prototype_label", pd.Series(dtype=str)))
    except (FileNotFoundError, ValueError, KeyError):
        # batches.xlsx missing/unreadable for this sheet — search still works
        # against Patent_ID alone.
        queue["Company"] = None
        queue["Architecture"] = None

    return queue.reset_index(drop=True)


def load_main_architecture_image(df: pd.DataFrame, patent_id: str, matched_dir: Path):
    """Best-effort lookup of the main architecture image for patent_id.

    Prefers the isMain-flagged T2 row's Image_Path as recorded in df; falls
    back to any T2 image for the patent, then to globbing matched_dir by the
    base patent id. Returns None (never raises) if nothing resolves.
    """
    t2_rows = df.loc[
        (df["Patent_ID"] == patent_id) & (df["Section"] == "T2") & df["Image_Path"].notna()
    ]
    image_path = None
    if not t2_rows.empty:
        is_main = t2_rows.loc[
            (t2_rows["Field"] == "isMain")
            & t2_rows["Value"].astype(str).str.lower().isin(["true", "1", "yes", "main"]),
            "Image_Path",
        ]
        image_path = (is_main.iloc[0] if not is_main.empty else t2_rows["Image_Path"].iloc[0])

    if image_path:
        candidate = Path(str(image_path))
        if candidate.exists():
            return candidate

    base_id, _ = proc.parse_arch_id(patent_id)
    try:
        matches = sorted(matched_dir.glob(f"{base_id}_*/*"))
    except OSError:
        matches = []
    return matches[0] if matches else None


class ReviewSession:
    """One-issue-at-a-time review widget over an issue_queue DataFrame."""

    def __init__(self, df: pd.DataFrame, issue_queue: pd.DataFrame, matched_dir: Path):
        self.df = df                 # mutated in place by the action buttons
        self.full_queue = issue_queue
        self.queue = issue_queue
        self.matched_dir = matched_dir
        self.pos = 0

        self.search_box = widgets.Text(
            description="Search:",
            placeholder="Filter by Company / Architecture / Patent ID",
            layout=widgets.Layout(width="480px"),
        )
        self.search_box.observe(self._on_search, names="value")

        self.status_label = widgets.Label()
        self.image_panel = widgets.Output(layout=widgets.Layout(width="45%", border="1px solid #ccc"))
        self.meta_panel = widgets.HTML(layout=widgets.Layout(width="55%"))

        self.prev_btn = widgets.Button(description="◀ Prev")
        self.next_btn = widgets.Button(description="Next ▶")
        self.cvt_btn = widgets.Button(description="Update to CVT (Combined)", button_style="warning")
        self.keep_btn = widgets.Button(description="Keep As Is", button_style="success")

        self.prev_btn.on_click(self._on_prev)
        self.next_btn.on_click(self._on_next)
        self.cvt_btn.on_click(self._on_cvt)
        self.keep_btn.on_click(self._on_keep)

        self.root = widgets.VBox([
            widgets.HBox([self.search_box]),
            self.status_label,
            widgets.HBox([self.image_panel, self.meta_panel]),
            widgets.HBox([self.prev_btn, self.next_btn, self.cvt_btn, self.keep_btn]),
        ])
        self._render()

    # ── navigation / filtering ──────────────────────────────────────────
    def _on_search(self, change):
        term = (change["new"] or "").strip().lower()
        if not term:
            self.queue = self.full_queue
        else:
            mask = self.full_queue.apply(
                lambda r: term in str(r.get("Company", "")).lower()
                or term in str(r.get("Architecture", "")).lower()
                or term in str(r["Patent_ID"]).lower(),
                axis=1,
            )
            self.queue = self.full_queue[mask].reset_index(drop=True)
        self.pos = 0
        self._render()

    def _on_prev(self, _btn):
        if self.pos > 0:
            self.pos -= 1
            self._render()

    def _on_next(self, _btn):
        if self.pos < len(self.queue) - 1:
            self.pos += 1
            self._render()

    def _current_patent_id(self):
        return None if self.queue.empty else self.queue.iloc[self.pos]["Patent_ID"]

    # ── decisions ────────────────────────────────────────────────────────
    def _on_cvt(self, _btn):
        """Reviewer confirms Combined Thrust: topType TP → CVT (the wizard's
        own composite value, so the file stays round-trip compatible)."""
        pid = self._current_patent_id()
        if pid is not None:
            mask = (self.df["Field"] == "topType") & (self.df["Patent_ID"] == pid)
            self.df.loc[mask, "Value"] = CVT_VALUE
            _append_flag(self.df, [pid], "Reviewer: Confirmed CVT (Combined Thrust);")
        self._advance()

    def _on_keep(self, _btn):
        pid = self._current_patent_id()
        if pid is not None:
            _append_flag(self.df, [pid], "Reviewer: Kept As Is;")
        self._advance()

    def _advance(self):
        if self.pos < len(self.queue) - 1:
            self.pos += 1
        self._render()

    # ── rendering ────────────────────────────────────────────────────────
    def _render(self):
        n = len(self.queue)
        if n == 0:
            self.status_label.value = "No flagged records match this filter."
            self.meta_panel.value = ""
            with self.image_panel:
                clear_output(wait=True)
            return

        self.pos = min(self.pos, n - 1)
        row = self.queue.iloc[self.pos]
        self.status_label.value = f"Reviewing {self.pos + 1} of {n}"
        self.meta_panel.value = (
            f"<b>Patent ID:</b> {row['Patent_ID']}<br>"
            f"<b>Company:</b> {row.get('Company') or '—'}<br>"
            f"<b>Architecture:</b> {row.get('Architecture') or '—'}<br>"
            f"<b>topType:</b> {row.get('topType') or '—'}<br>"
            f"<b>Flags:</b> {row.get('flags') or '—'}"
        )

        with self.image_panel:
            clear_output(wait=True)
            try:
                image_path = load_main_architecture_image(self.df, row["Patent_ID"], self.matched_dir)
            except Exception as exc:  # never let a lookup error break the UI
                print(f"⚠ image lookup failed for {row['Patent_ID']}: {exc}")
                image_path = None

            if image_path is None:
                print(f"⚠ no image found for {row['Patent_ID']}")
            else:
                try:
                    display(widgets.Image(
                        value=image_path.read_bytes(),
                        format=image_path.suffix.lstrip(".") or "png",
                        layout=widgets.Layout(max_width="100%"),
                    ))
                except Exception as exc:
                    print(f"⚠ could not load image {image_path}: {exc}")

    def display(self):
        display(self.root)


def build_review_grid(df: pd.DataFrame, issue_queue: pd.DataFrame, matched_dir: Path = matched_dir) -> ReviewSession:
    session = ReviewSession(df, issue_queue, matched_dir)
    session.display()
    return session


# issue_queue = build_issue_queue(df, cfg, sheet_name)
# session = build_review_grid(df, issue_queue)


## Section 6 — Export

Writes the cleaned `df` through `format_review_workbook` (Section 2) into a
**brand-new** file — never `REVIEWED_XLSX` (the raw `01a_review` export) and
never a previous preprocessing run's output. Safety net: a timestamped
filename in its own `preprocessed/` subfolder under `html_review_exports`,
plus an explicit guard against collisions before writing.


In [ ]:
OUTPUT_DIR = Path(cfg["paths"]["html_review_exports"]) / "preprocessed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_XLSX = OUTPUT_DIR / f"preprocessed_patents_{sheet_name}_{timestamp}.xlsx"

# Guard against ever clobbering the raw wizard export or a prior run's output.
assert OUTPUT_XLSX != REVIEWED_XLSX, "refusing to overwrite the raw 01a_review export"
assert not OUTPUT_XLSX.exists(), f"{OUTPUT_XLSX} already exists — refusing to overwrite"

format_review_workbook(df, OUTPUT_XLSX, truncate_long_text=False)

print(f"wrote cleaned batch to: {OUTPUT_XLSX}")
